LIBRARIES

In [ ]:
%matplotlib inline

import gc
import json
import os
import re
import random
import shutil
import time
import tempfile
import subprocess
from collections import defaultdict
from glob import glob
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from loguru import logger
from tqdm import tqdm
from ultralytics import YOLO
from PIL import Image

PARAMETERS
(Make changes here before running the workflow)

In [ ]:
# GLOBAL PARAMS
images_csv = 'images_2007780.csv'
deployments_csv = 'deployments.csv'
images_exif_csv = 'images_with_exif_camera_details.csv'
# pickup_date = '2024-03-23'  # wont be needed if human images were manually extracted
dataset_name = 'snapshotusa24_project'
device = 'cuda'
#cls_model_weights = 'yolov8x.pt'  # consider: 'yolov8x-worldv2.pt'
cls_model_weights = 'signs_best.pt'

In [ ]:
assert all([Path(x).exists() for x in [images_csv, deployments_csv, cls_model_weights]])

DEFINED FUNCTIONS
(DO NOT change anything, only add if needed)

In [ ]:
def change_file_time(file, t):
    timestamp_str = str(t)
    parsed_time = time.strptime(timestamp_str, '%Y-%m-%d %H:%M:%S')
    ts = time.mktime(parsed_time)
    os.utime(file, (ts, ts))

In [ ]:
def visualize_contours(image, binary_mask):
    contours, _ = cv2.findContours(binary_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contour_image = image.copy()
    cv2.drawContours(contour_image, contours, -1, (255,0,0), 3)
    plt.figure(figsize=(len(image[0]) / 100.0, len(image) / 100.0))  # Adjusting figure size
    plt.imshow(contour_image)
    plt.axis('off')
    plt.show()

In [ ]:
def calculate_area(bbox):
    xmin, ymin, xmax, ymax = bbox
    width = xmax - xmin
    height = ymax - ymin
    area = width * height
    return area

In [ ]:
def get_bbox_image(im_path, bbox):
    image = cv2.imread(im_path)
    mask = np.zeros_like(image)
    x1, y1, x2, y2 = [int(coord) for coord in bbox]
    mask[y1:y2, x1:x2] = [255, 255, 255]
    output_image_path = f'{deployment_path}/calibration_frames_masks/{Path(im_path).name}'
    cv2.imwrite(output_image_path, mask)

    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 3)
    # Convert BGR to RGB for matplotlib display since OpenCV uses BGR by default.
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Display the image inline in Jupyter.
    # plt.figure(figsize=(10, 10))  # You can adjust the figure size as needed.
    # plt.imshow(image_rgb)  
    # plt.axis('off')  # Turn off axis numbers and ticks.
    # plt.show()

In [ ]:
# Image is already cropped when specifying the flag --crop*; not needed.
# def crop_image(camera_name, image_path, save_path):
#     image = cv2.imread(image_path)
#     height, width = image.shape[:2]
#     crop_val = 0.09
#     crop_height = int(height * crop_val)

#     # Adjust crop_height based on camera name
#     if 'Reconyx' in camera_name:
#         crop_height *= 2  # Double the crop height for Reconyx cameras

#     # Ensure crop indices are within valid range
#     end_row = min(height, height - crop_height)  # Changed to keep the top part

#     # Crop the image from the bottom
#     image = image[:end_row, :]  # Changed to keep the top part

#     cv2.imwrite(save_path, image)

In [ ]:
# for testing on how much to crop

def crop_from_top_and_bottom(image_path, pixels_to_crop_top, pixels_to_crop_bottom):
    img = Image.open(image_path)
    width, height = img.size

    new_height = height - pixels_to_crop_top - pixels_to_crop_bottom

    if new_height <= 0:
        raise ValueError("Cannot crop more pixels than the image height.")

    left = 0
    top = pixels_to_crop_top  # Start cropping from the top
    right = width
    bottom = height - pixels_to_crop_bottom  # Stop cropping from the bottom

    cropped_img = img.crop((left, top, right, bottom))
    display(cropped_img)

In [ ]:
def clean_deployment_id(s):
    return (
        s.str.strip()
         .str.replace('/', '-', regex=False)
         .str.replace(' ', '-', regex=False)
         .str.replace(',', '-', regex=False)
         .str.replace('.', '-', regex=False)
         .str.replace(':', '-', regex=False)
    )

def has_files(path: Path):
    try:
        return any(f.is_file() for f in path.iterdir())
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return False

In [ ]:
# sample_image = 'mm2/Reconyx Hyperfire 1/transects/B0/detection_frames/7db79634-6680-4bd7-8dc0-c9fb97d90113.JPG'
# crop_from_top_and_bottom(sample_image, 35, 65)

In [ ]:
df = pd.read_csv(images_csv)
df_deployments = pd.read_csv(deployments_csv)
df_images_exif = pd.read_csv(images_exif_csv)

In [ ]:
len(df)

In [ ]:
df['deployment_id'] = clean_deployment_id(df['deployment_id'])
df_deployments['deployment_id'] = clean_deployment_id(df_deployments['deployment_id'])
df_images_exif['deployment_id'] = clean_deployment_id(df_images_exif['deployment_id'])

df = df[df["common_name"] != "Blank"]

In [ ]:
len(df)

In [ ]:
existing_sorted_transects_paths = glob(f'{dataset_name}/*/transects/*')
existing_sorted_transects_ids = [Path(x).name for x in existing_sorted_transects_paths]
len(existing_sorted_transects_ids)

In [ ]:
animal_images_local_ids = [Path(x).stem for x in glob('animal_images/*')]
len(animal_images_local_ids)

In [ ]:
df_of_existing_deployments = df[df.deployment_id.isin(existing_sorted_transects_ids)]
len(df_of_existing_deployments.deployment_id.unique())

In [ ]:
df = df_of_existing_deployments

In [ ]:
df

In [ ]:
# # exclude folders without images or with only 1 image
# human_dplys = glob('human_images/*')
# exclude_human_dplys = []
# print(len(human_dplys))
# for folder in human_dplys:
#     if len(glob(f'{folder}/*')) < 2:
#         human_dplys.remove(folder)
#         exclude_human_dplys.append(folder)
# print(len(human_dplys))
# human_dplys = [Path(x).stem for x in human_dplys]

In [ ]:
# Path('excluded_human_images').mkdir(exist_ok=True)
# for folder in exclude_human_dplys:
#     shutil.move(folder, 'excluded_human_images')

In [ ]:
# print(len(df))
# df = df[df['deployment_id'].isin(human_dplys)]
# df_deployments = df_deployments[df_deployments['deployment_id'].isin(human_dplys)]
# df_images_exif = df_images_exif[df_images_exif['deployment_id'].isin(human_dplys)]
# len(df.deployment_id.unique())

In [ ]:
# missing_from_df = []
# deployment_ids = df.deployment_id.unique()
# for x in human_dplys:
#     if x not in deployment_ids:
#         missing_from_df.append(x)
# with open('in_human_images_but_missing_from_images_csv.json', 'w') as j:
#     json.dump(missing_from_df, j, indent=4)
# len(missing_from_df)

In [ ]:
df = df.merge(df_deployments[['camera_name', 'deployment_id']], on='deployment_id', how='inner')
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [ ]:
df

In [ ]:
print('df length:', len(df))
print('Number of deployments in df:', len(df['deployment_id'].unique()))

_df = df.copy()
df_animals = _df[~_df['common_name'].isin(['Human-Camera Trapper', 'Human-Faces', 'Human', 'Blank', 'Calibration distance', 'Vehicle', 'Human-Pedestrian', 'Human - Biker'])]

In [ ]:
df_animals = df_animals.sort_values(by='timestamp')
df_animals.reset_index(drop=True, inplace=True)
df_animals = df_animals.drop_duplicates()

print('Animals df length:', len(df_animals))

In [ ]:
# df_animals['location'].to_csv('animal_images.csv', index=False, header=False)
# Path('animal_images').mkdir(exist_ok=True)

# ! cat 'animal_images.csv' | gsutil -m cp -I 'animal_images'

In [ ]:
# check if deployment with calibrations all exist in animals_df
print(len(human_dplys))
human_dplys = [Path(x).name for x in human_dplys]
len(df_animals[df_animals.deployment_id.isin(human_dplys)].deployment_id.unique())

In [ ]:
# only include deployments with calibrations
df_animals = df_animals[df_animals.deployment_id.isin(human_dplys)]
len(df_animals)

In [ ]:
df['image_stem'] = df['location'].apply(
    lambda x: Path(str(x)).stem if pd.notnull(x) else None
)

In [ ]:
# Download images missing from `animal_images`
animal_stems = [Path(x).stem for x in glob('animal_images/*')]
to_download = df[~df.image_id.isin(animal_stems)]
# to_download['location'].to_csv('download_now.csv', header=False, index=False)

In [ ]:
# lookup = dict(zip(df["image_stem"].astype(str), df["image_id"].astype(str)))

# # Folder with images
# folder = Path("animal_images")

# for file in folder.iterdir():
#     if file.is_file():
#         stem = file.stem
#         if stem in lookup:
#             new_name = lookup[stem] + file.suffix  # keep same extension
#             new_path = file.with_name(new_name)
#             # Only rename if not already correct
#             if file.name != new_name:
#                 print(f"Renaming {file.name} -> {new_name}")
#                 file.rename(new_path)

In [ ]:
# SKIP THIS STEP IF ANIMAL_IMAGES ALREADY EXIST AND IS SORTED BY DEPLOYMENT

image_doesnt_exist = []

for x in df_animals['deployment_id'].unique():
    Path(f'animal_images/{x}').mkdir(exist_ok=True, parents=True)

for x, y, z, t in zip(df_animals['location'], df_animals['deployment_id'], df_animals['image_id'], df_animals['timestamp']):
    out_path = f'animal_images/{y}/{z}{Path(x).suffix}'
    if Path(out_path).exists():
        continue
    try:
        if not Path(Path(x).name).exists():
            x = f'animal_images/{z}{Path(x).suffix}'
        shutil.move(f'animal_images/{Path(x).name}', out_path)
        change_file_time(out_path, t)
    except Exception as e:
        image_doesnt_exist.append(x)


In [ ]:
len(image_doesnt_exist)

In [ ]:
print(len(image_doesnt_exist))
if image_doesnt_exist:
    pd.DataFrame(image_doesnt_exist).to_csv('missing_animal_images.csv', header=False, index=False)

In [ ]:
animal_dplys = glob('animal_images/*')
print(len(animal_dplys))
for folder in animal_dplys:
    if not glob(f'{folder}/*'):
        animal_dplys.remove(folder)
print(len(animal_dplys))
animal_dplys = [Path(x).stem for x in animal_dplys]

In [ ]:
df = df[df.deployment_id.isin(animal_dplys)]

In [ ]:
# We moved images that don't belong to any existing deployment in the filtered deployments set
# for x in glob('animal_images/*.jpg') + glob('animal_images/*.JPG'):
#     shutil.move(x, 'unsorted_animal_images')
# [x for x in glob('animal_images/*') if Path(x).is_file()]

In [ ]:
# ignoring since we organized manually
# print(f'Cameras pick-up date: {pickup_date}')
# df_human = df[df['timestamp'].dt.date == pd.to_datetime(pickup_date).date()]

# df_human = df[df['common_name'].isin(['Calibration distance'])]
# print('Human df length:', len(df_human))

# df_human['location'].to_csv('human_images.csv', index=False, header=False)
# Path('human_images').mkdir(exist_ok=True)

# ! cat 'human_images.csv' | gsutil -m cp -I 'human_images'

In [ ]:
# not needed when human images are moved manually into deployment folders

# for x in df_human['deployment_id'].unique():
#     Path(f'human_images/{x}').mkdir(exist_ok=True, parents=True)

# for x, y, z, t in zip(df_human['location'], df_human['deployment_id'], df_human['image_id'], df_human['timestamp']):
#     out_path = f'human_images/{y}/{z}{Path(x).suffix}'
#     shutil.move(f'human_images/{Path(x).name}', out_path)
#     change_file_time(out_path, t)

In [ ]:
len(df)

In [ ]:
human = [Path(p).name for p in glob('human_images/*')]
animal = [Path(p).name for p in glob('animal_images/*')]

diff_a = [x for x in animal if x not in human]
diff_h = [x for x in human if x not in animal]

print("In animal but not in human:", diff_a, len(diff_a))
print("In human but not in animal:", diff_h, len(diff_h))

assert not diff_a
assert not diff_h

In [ ]:
Path('animal_images_in_animal_but_not_in_human').mkdir(exist_ok=True)
for folder in diff_a:
    shutil.move(f'animal_images/{folder}', 'animal_images_in_animal_but_not_in_human')

In [ ]:
Path('in_human_but_not_in_animal').mkdir(exist_ok=True)
for folder in diff_h:
    shutil.move(f'human_images/{folder}', 'in_human_but_not_in_animal')

In [ ]:
missing_subset = df[df.deployment_id.isin(diff_h)]
print(len(missing_subset))
if len(missing_subset) != 0:
    missing_subset['location'].to_csv('missing_animal_images.csv', index=False, header=False)

In [ ]:

# #---------------------------------------------------
# # WARNING !
# # SPECIFIC TO ONE DATASET, DON'T USE IN PRODUCTION
# mapping = {
#     'HC500 Hyperfire Semi-Covert IR': 'Reconyx HC500 Hyperfire',
#     'HC500 Hyperfire Semi-Covert IR_': 'Reconyx HC500 Hyperfire',
#     'Reconyx HC500 Hyperfire': 'Reconyx Hyperfire 1',
#     "Reconyx": "Reconyx Hyperfire 1",
#     "reconyx HC500": "Reconyx Hyperfire 1",
#     "Browning": "Browning Recon Force Elite",
#     "BrowningRecon Force Elite": "Browning Recon Force Elite",
#     "0": "Bushnell",
#     "BAC.TRMR_Rec041_Mem086.20240910_Reconyx Hyperfire HC500": "Reconyx Hyperfire 1",
#     "Bushnell DS-4K": "Bushnell",
#     "Bushnell Core DS-4K": "Bushnell",
#     "Reconyx Hyperfire HC500": "Reconyx Hyperfire 1",
# }
# df['camera_name'] = df['camera_name'].replace(mapping)
# # WARNING !
# #---------------------------------------------------


In [ ]:
# from pathlib import Path

# df["image_local_path"] = df.apply(
#     lambda row: f'animal_images/{row["deployment_id"]}/{Path(row["location"]).name}',
#     axis=1
# )

In [ ]:
df_images_exif["camera_make_model"] = (
    df_images_exif["camera_make"].astype(str).str.strip() + " " +
    df_images_exif["camera_model"].astype(str).str.strip())

df = df.merge(df_images_exif[["deployment_id", "camera_make_model"]],
              on="deployment_id",
              how="left")

df.loc[df["camera_make_model"].notna(),
       "camera_name"] = df["camera_make_model"]

df = df.drop(columns=["camera_make_model"])

In [ ]:
len(df['camera_name'].unique())

In [ ]:
# pattern = re.compile(r'(Reconyx.*|Browning.*|Bushnell.*|BTC-5HDX|PC800|HC500|HC600|HP2X)')

# def extract_camera_model(name):
#     match = pattern.search(name)
#     return match.group(0).strip() if match else name

# df["camera_name_normalized"] = df["camera_name"].apply(extract_camera_model)

In [ ]:
# len(df["camera_name_normalized"].unique())

In [ ]:
df['camera_name'].unique()

In [ ]:
for camera_name in df['camera_name'].unique():
    Path(f'{dataset_name}/{camera_name}/transects').mkdir(exist_ok=True, parents=True)
    Path(f'{dataset_name}/{camera_name}/results').mkdir(exist_ok=True)

for deployment in human:
    if len(df[df['deployment_id'] == deployment]) == 0:
        continue
    camera_name = df[df['deployment_id'] == deployment]['camera_name'].unique()[0]

    for subfolder in ['calibration_frames', 'calibration_frames_masks', 'detection_frames']:
        Path(f'{dataset_name}/{camera_name}/transects/{deployment}/{subfolder}').mkdir(exist_ok=True, parents=True)
    

In [ ]:
for deployment in tqdm(animal):
    deployment_path = f'animal_images/{deployment}'
    # deployment_name = Path(deployment).name
    camera_name = df[df['deployment_id'] == deployment]['camera_name'].unique()[0]
    out_path = f'{dataset_name}/{camera_name}/transects/{Path(deployment).name}/detection_frames'
    if len(glob(f'{deployment_path}/*')) == len(glob(f'{out_path}/*')):
        continue
    for image in glob(f'{deployment_path}/*'):
        # if Path(f'{image}/{Path(image).name}').exists():
        #     continue
        shutil.copy2(image, out_path)

In [ ]:
for deployment in tqdm(human):
    deployment_path = f'human_images/{deployment}'
    # deployment_name = Path(deployment).name
    try:
        camera_name = df[df['deployment_id'] == deployment]['camera_name'].unique()[0]
    except:
        print(deployment)
        # break
    out_path = f'{dataset_name}/{camera_name}/transects/{Path(deployment).name}/calibration_frames'
    # if not Path(out_path).exists():
    #     continue
    for image in glob(f'{deployment_path}/*'):
        shutil.copy2(image, out_path)

In [ ]:
transects = glob(f'{dataset_name}/*/transects/*')
print(len(transects))
# transects

In [ ]:
transects[0]

In [ ]:
model = YOLO(cls_model_weights)

In [ ]:
dply_names = [Path(x).name for x in transects]
dply_dict = {x: [] for x in dply_names}

In [ ]:
failed_signs = []
for transect in tqdm(transects):
    dply_name = Path(transect).name
    for image_path in glob(f'{transect}/calibration_frames/*'):
        n = Path(image_path).stem
        res = model(image_path)
        try:
            d = {n: res[0].boxes.xyxy[0].tolist()}
        except IndexError:
            failed_signs.append(image_path)
            continue
        try:
            dply_dict[dply_name].append(d)
        except KeyError:
            print(f'Deployment {dply_name} does not exist!')

In [ ]:
failed_signs

In [ ]:
# BBOX MASKS

failed_images = []
failed_dply_boxes = []

for deployment_path in tqdm(transects):        
    ims = glob(f'{deployment_path}/calibration_frames/*')
    ims.sort()
    for im_path in ims:
        dply = Path(im_path).parent.parent.name
        dply_boxes = dply_dict[dply]
        n = Path(im_path).stem
        try:
            bbox = [x for x in dply_boxes if x.get(n)][0][n]
        except IndexError:
            failed_dply_boxes.append(im_path)
            continue
        failed_image_path = get_bbox_image(im_path, bbox)
        if failed_image_path:
            logger.warning(f'Failed to extract mask from: {failed_image_path}')
            # Path(im_path).unlink()
            failed_images.append(im_path)

In [ ]:
failed_dply_boxes

In [ ]:
failed_images

In [ ]:
for camera_name in df['camera_name'].unique():
    parents = glob(f'{dataset_name}/{camera_name}/transects/*')
    for x in parents:
        l1 = len(glob(f'{x}/calibration_frames/*'))
        l2 = len(glob(f'{x}/calibration_frames_masks/*'))
        assert l1 == l2

In [ ]:
with open('cameras_specs.json') as j:
    camera_specs = json.load(j)

In [ ]:
project_per_camera = glob(f'{dataset_name}/*')
project_per_camera

In [ ]:
camera_specs

In [ ]:
# del predictor
# del sam
gc.collect()
torch.cuda.empty_cache()

In [ ]:
camera_crop_mapping = {
    "Browning Recon Force Elite": {
        "crop_top": 0,
        "crop_bottom": 100
    },
    "Reconyx Hyperfire 1": {
        "crop_top": 35,
        "crop_bottom": 65
    },
    "Bushnell": {
        "crop_top": 0,
        "crop_bottom": 106
    }
}

In [ ]:
with open('camera_crop_mapping.json', 'w') as j:
    json.dump(camera_crop_mapping, j, indent=4)

```bash
export LD_LIBRARY_PATH=/home/biodiv/mambaforge/envs/dnd/lib/python3.11/site-packages/nvidia/cudnn/lib:$LD_LIBRARY_PATH
```


```bash
python main.py \
    --cli \
    --bbox_confidence_threshold 0.60 \
    --bbox_sampling_percentile 20 \
    --calibration_regression_method "ransac" \
    --camera_horizontal_fov 41.0 \
    --camera_vertical_fov 41.0 \
    --detection_sampling_method "bbox_percentile" \
    --sample_from "detection" \
    --data_dir "../mm2/Browning Recon Force Elite" \
    --draw_world_position \
    --draw_detection_ids \
    --crop_bottom 100 \
    --depth_estimation_model 'dpt_pytorch'
```

```bash
 python main.py \
    --cli \
    --bbox_confidence_threshold 0.60 \
    --bbox_sampling_percentile 20 \
    --calibration_regression_method "ransac" \
    --camera_horizontal_fov 42.2 \
    --camera_vertical_fov 32.3 \
    --detection_sampling_method "bbox_percentile" \
    --sample_from "detection" \
    --data_dir "../mm2/Reconyx Hyperfire 1" \
    --draw_world_position \
    --draw_detection_ids \
    --crop_bottom 65 \
    --crop_top 35 \
    --depth_estimation_model 'dpt_pytorch'
```

In [ ]:
# Move results under 'sampling' to their respective deployment
for camera_name in glob(f'{dataset_name}/*'):
    camera_name = Path(camera_name).name
    transects = glob(f'{dataset_name}/{camera_name}/transects/*')

    data = {}
    for t in transects:
        files = glob(f'{t}/detection_frames/*')
        data[Path(t).name] = [Path(x).stem for x in files]

    for k, v in data.items():
        parent_dir = f'{dataset_name}/{camera_name}/results/sampling'
        output_dir = f'{parent_dir}/{k}'
        Path(output_dir).mkdir(exist_ok=True)
        for name in v:
            try:
                shutil.move(f'{parent_dir}/{name}.jpg', output_dir)
            except FileNotFoundError:
                continue

In [ ]:
for camera_name in glob(f'{dataset_name}/*'):
    print(camera_name)
    results_df = pd.read_csv(f'{camera_name}/results/results.csv')
    results_df['image_id'] = results_df['frame_id']
    results_df = pd.merge(df, results_df, on='image_id', how='inner')
    results_df.to_csv(f'{camera_name}/results/results-{Path(camera_name).name}-full.csv', index=False)

In [ ]:
parent_dir = f'{dataset_name}/*/results/calibration_arrays'
arrs = glob(f'{parent_dir}/*')

rows = []
for arr_path in arrs:
    arr = np.load(arr_path)
    camera_name = Path(arr_path).parents[2].name
    row = [Path(arr_path).stem.rstrip('_calibration'), camera_name, arr.mean(), np.median(arr), arr.min(), arr.max()]
    rows.append(row)

df_mmm = pd.DataFrame(rows, columns=['deployment_id', 'camera_name', 'mean', 'median', 'min', 'max'])
df_mmm.to_csv(f'{dataset_name}/depth_mmm.csv', index=False)
df_mmm